Notebook for testing different methods on the LOOCV folds. The results were stored and saved on Google Drive.

In [ ]:
import sys
!git clone https://github.com/Ignas12345/masters_project_helper_functions.git
sys.path.append('/content/masters_project_helper_functions')

Cloning into 'masters_project_helper_functions'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 196 (delta 15), reused 20 (delta 9), pack-reused 169 (from 1)
Receiving objects: 100% (196/196), 75.70 KiB | 745.00 KiB/s, done.
Resolving deltas: 100% (114/114), done.


In [ ]:
mount_drive = 1
if mount_drive:
    from google.colab import drive
    drive.mount('/content/Gdrive')

Mounted at /content/Gdrive


In [ ]:
import pandas as pd
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV

import masters_project_helper_functions.utils as utils
import masters_project_helper_functions.plotting as plotting
import masters_project_helper_functions.preprocessing_methods as pp
import masters_project_helper_functions.feature_selection as feature_selection
import masters_project_helper_functions.classification_pipeline as classification_pipeline

In [ ]:
url_TCGA_mirna_raw_counts = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TCGA_TGCT_mirna_isoform_data_raw_counts.csv"
url_TCGA_mirna_rpm_counts = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TCGA_TGCT_mirna_isoform_data_rpm.csv"
url_TCGA_TGCT_divisions_by_experiment = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/sample_annotations/TCGA_TGCT_divisions_by_experiment.csv"
fold_path = 'https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/bootstrap_folds/'

TCGA_TGCT_divisions_by_experiment = pd.read_csv(url_TCGA_TGCT_divisions_by_experiment, index_col=0)
TCGA_mirna_rpm_counts = utils.format_data_frame(url_TCGA_mirna_rpm_counts, sep = ';', decimal= ',', transpose = True).loc[TCGA_TGCT_divisions_by_experiment.index]

TCGA_mirna_rpm_counts = utils.initial_pre_processing_pipeline(TCGA_mirna_rpm_counts)
experiment_label_dict = TCGA_TGCT_divisions_by_experiment.to_dict()
fold_path = 'https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/bootstrap_folds/'

reading df from url: https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TCGA_TGCT_mirna_isoform_data_rpm.csv
nan values filled with 0
column names truncated using: slice(None, 15, None)
df transposed
final shape of df: (156, 3689)

initial shape: (137, 3689)
shape after collapsing: (137, 2212)
initial shape: (137, 2212)
shape after filtering: (137, 2191)


In [ ]:
results_df_url = '/content/Gdrive/MyDrive/Magistro_projektas/loocv_results/results_df.csv'

try:
  results_df = pd.read_csv(results_df_url, sep = ';', index_col = 0)
except:
  results_df = pd.DataFrame()
display(results_df)

In [ ]:
df = results_df[results_df['experiment_name'] == 'seminoma_vs_embryonal'].sort_values('matthews_corrcoef', ascending = False).copy()
display(df)

,experiment_name,initial_feature_filtering,sample_wise_scaling,feature_wise_scaling,feature_selection_method,classification_method,housekeeping_list,scale_housekeep_by_mean,classification_method_parameters,feature_ranking_method,...,mean_feature_number,balanced_accuracy_score,recall_score,precision_score,brier_score_loss,matthews_corrcoef,rank_features_and_keep_top_n,drop_correlated_features,estimator_for_rfe_ranking,time_taken
45,seminoma_vs_embryonal,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,select_all_features,LinearSVC,"['hsa-mir-191, mature,MIMAT0000440']",False,"{'C': 100, 'class_weight': 'balanced', 'penalt...",NaN,...,14.533333,1.000000,1.000000,1.000000,0.039065,1.000000,NaN,NaN,NaN,40.321103
25,seminoma_vs_embryonal,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,rfecv_feature_selection,LogisticRegression,"['hsa-mir-191, mature,MIMAT0000440']",False,"{'C': 100, 'class_weight': 'balanced'}",perform_RFE_ranking,...,3.255556,0.972222,0.944444,1.000000,0.010381,0.965146,rank_features_and_keep_top_n_features,NaN,"LogisticRegression(C=100, class_weight='balanc...",139.900486
37,seminoma_vs_embryonal,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,select_all_features,AdaBoostClassifier,"['hsa-mir-191, mature,MIMAT0000440']",False,{},NaN,...,1.000000,0.972222,0.944444,1.000000,0.022671,0.965146,NaN,NaN,NaN,35.914233
31,seminoma_vs_embryonal,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,forward_feature_selection_cv,SVC,"['hsa-mir-191, mature,MIMAT0000440']",False,"{'C': 100, 'class_weight': 'balanced'}",NaN,...,1.000000,0.972222,0.944444,1.000000,0.019904,0.965146,NaN,NaN,NaN,490.021392
43,seminoma_vs_embryonal,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,rfecv_feature_selection,ExtraTreesClassifier,"['hsa-mir-191, mature,MIMAT0000440']",False,{'class_weight': 'balanced'},perform_RFE_ranking,...,1.388889,0.972222,0.944444,1.000000,0.011389,0.965146,rank_features_and_keep_top_n_features,NaN,ExtraTreesClassifier(class_weight='balanced'),5168.370830
51,seminoma_vs_embryonal,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,rfecv_feature_selection,LinearSVC,"['hsa-mir-191, mature,MIMAT0000440']",False,"{'C': 100, 'class_weight': 'balanced', 'penalt...",perform_RFE_ranking,...,3.177778,0.972222,0.944444,1.000000,0.052270,0.965146,rank_features_and_keep_top_n_features,NaN,"LinearSVC(C=100, class_weight='balanced', max_...",576.444171
54,seminoma_vs_embryonal,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,rfecv_feature_selection,LinearSVC,"['hsa-mir-191, mature,MIMAT0000440']",False,"{'C': 100, 'class_weight': 'balanced', 'penalt...",perform_RFE_ranking,...,3.066667,0.972222,0.944444,1.000000,0.048875,0.965146,rank_features_and_keep_top_n_features,NaN,"LinearSVC(C=100, class_weight='balanced', max_...",595.595384
48,seminoma_vs_embryonal,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,rfecv_feature_selection,LinearSVC,"['hsa-mir-191, mature,MIMAT0000440']",False,"{'C': 100, 'class_weight': 'balanced', 'penalt...",perform_RFE_ranking,...,3.000000,0.965278,0.944444,0.944444,0.047450,0.930556,rank_features_and_keep_top_n_features,NaN,"LinearSVC(C=100, class_weight='balanced', max_...",783.925539


In [ ]:
pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,
                          }
feature_selection_method = feature_selection.rfecv_feature_selection

feature_ranking_method = pp.perform_RFE_ranking


classification_method = LinearSVC
possible_parameters = [{'C' : 100, 'class_weight' : 'balanced', 'penalty' : 'l1', 'max_iter' : 100000}]

'''
classification_method = LinearSVC
possible_parameters = [{'C' : 100, 'class_weight' : 'balanced', 'probability' : True}]
'''

"\nclassification_method = LinearSVC\npossible_parameters = [{'C' : 100, 'class_weight' : 'balanced', 'probability' : True}]\n"

In [ ]:
#experiments = ['seminoma_vs_non_seminoma', 'seminoma_vs_embryonal', 'embryonal_vs_non_embryonal_non_seminoma', 'teratoma_vs_non_teratoma', 'KIT_wildtype_vs_mutated_seminoma', 'teratoma_present_vs_teratoma_absent']
experiments = ['seminoma_vs_non_seminoma','seminoma_vs_embryonal', 'embryonal_vs_non_embryonal_non_seminoma',]
for experiment_name in experiments:
  print(experiment_name)
  sample_label_dict = experiment_label_dict[experiment_name]
  df = TCGA_mirna_rpm_counts.copy()

  kwargs = {'housekeeping_list' : ['hsa-mir-191, mature,MIMAT0000440', ],
          'scale_housekeep_by_mean' : False,
          'feature_ranking_method' : feature_ranking_method,
          'keep_n_ranked_features' : 5,
          #'drop_correlated_features': True
          }

  kwargs['classification_method_parameters'] = possible_parameters[0]

  result, detailed_result = classification_pipeline.run_pipeline_on_loocv_folds_and_record_performance(df, fold_path=fold_path, experiment_name=experiment_name, sample_label_dict=sample_label_dict, pre_processing_methods=pre_processing_methods,
                                                                       feature_selection_method = feature_selection_method, classification_method = classification_method, **kwargs)

  results_df.to_csv(f'/content/Gdrive/MyDrive/Magistro_projektas/loocv_results/results_df_back_up.csv', sep = ';')
  results_df = pd.concat([results_df, result]).reset_index(drop=True)
  results_df.to_csv(f'/content/Gdrive/MyDrive/Magistro_projektas/loocv_results/results_df.csv', sep = ';')
  display(results_df)


In [ ]:
pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,}

feature_selection_method = feature_selection.rfecv_feature_selection

feature_ranking_method = pp.perform_RFE_ranking

classification_method = ExtraTreesClassifier
possible_parameters = [{'class_weight' : 'balanced'}]

In [ ]:
experiments = ['seminoma_vs_non_seminoma', 'seminoma_vs_embryonal', 'embryonal_vs_non_embryonal_non_seminoma', 'teratoma_vs_non_teratoma', 'KIT_wildtype_vs_mutated_seminoma', 'teratoma_present_vs_teratoma_absent']
for experiment_name in experiments:
  print(experiment_name)
  sample_label_dict = experiment_label_dict[experiment_name]
  df = TCGA_mirna_rpm_counts.copy()

  kwargs = {'housekeeping_list' : ['hsa-mir-191, mature,MIMAT0000440', ],
          'scale_housekeep_by_mean' : False,
          'feature_ranking_method' : feature_ranking_method,
          'keep_n_ranked_features' : 15,
          #'drop_correlated_features': True
          }

  kwargs['classification_method_parameters'] = possible_parameters[0]

  result, detailed_result = classification_pipeline.run_pipeline_on_loocv_folds_and_record_performance(df, fold_path=fold_path, experiment_name=experiment_name, sample_label_dict=sample_label_dict, pre_processing_methods=pre_processing_methods,
                                                                       feature_selection_method = feature_selection_method, classification_method = classification_method, **kwargs)

  results_df.to_csv(f'/content/Gdrive/MyDrive/Magistro_projektas/loocv_results/results_df_back_up.csv', sep = ';')
  results_df = pd.concat([results_df, result]).reset_index(drop=True)
  results_df.to_csv(f'/content/Gdrive/MyDrive/Magistro_projektas/loocv_results/results_df.csv', sep = ';')
  display(results_df)

In [ ]:
display(detailed_result)
result

,prediction,prob_class_1,true label,number_of_features_used
TCGA-2G-AAEW-01,0,0.0,0,2.0
TCGA-2G-AAEX-01,0,0.0,0,2.0
TCGA-2G-AAF1-01,0,0.0,0,2.0
TCGA-2G-AAF4-01,0,0.0,0,2.0
TCGA-2G-AAF6-01,0,0.0,0,2.0
...,...,...,...,...
TCGA-ZM-AA0D-01,0,0.0,0,2.0
TCGA-ZM-AA0E-01,0,0.09,0,3.0
TCGA-ZM-AA0F-01,0,0.0,0,2.0
TCGA-ZM-AA0H-01,0,0.0,0,2.0


,experiment_name,initial_feature_filtering,sample_wise_scaling,rank_features_and_keep_top_n,feature_wise_scaling,feature_selection_method,classification_method,housekeeping_list,scale_housekeep_by_mean,classification_method_parameters,...,smaller_group_label,larger_group_label,classifier_for_feature_selection,accuracy_score,mean_feature_number,balanced_accuracy_score,recall_score,precision_score,brier_score_loss,matthews_corrcoef
0,teratoma_vs_non_teratoma,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,forward_feature_selection_cv,ExtraTreesClassifier,"[hsa-mir-16-1, mature,MIMAT0000069]",False,{'class_weight': 'balanced'},...,teratoma,non_teratoma,ExtraTreesClassifier(class_weight='balanced'),0.964602,1.973451,0.745327,0.5,0.75,0.033481,0.595388
